# Busca Sequencial: Melhor Caso, Pior Caso e Caso Médio

Este notebook analisa e compara graficamente as três eficiências da **busca sequencial**:

| Caso | Condição | Custo |
|------|----------|-------|
| **Melhor caso** | item na 1ª posição | $C_{best}(n) = 1$ |
| **Pior caso** | item ausente ou na última posição | $C_{worst}(n) = n$ |
| **Caso médio** | busca bem-sucedida equiprovável | $C_{avg}(n) = \dfrac{p(n+1)}{2} + n(1-p)$ |

Para $p = 1$ (busca sempre bem-sucedida): $C_{avg}(n) = \dfrac{n+1}{2}$

---

## 1) Configuração e importações

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")

# Tamanhos de entrada avaliados
TAMANHOS = list(range(1, 501))

# Probabilidade de busca bem-sucedida usada no caso médio
P = 1.0   # p=1 → C_avg = (n+1)/2  ;  altere para ver outros cenários

## 2) Busca sequencial instrumentada

A função retorna o índice encontrado **e** o número de comparações realizadas (operação básica do algoritmo).

In [ ]:
def busca_sequencial(arr, chave):
    """
    Busca sequencial com contagem da operação básica (comparação A[i] != chave).
    Retorna (índice_encontrado_ou_-1, número_de_comparações).
    """
    comparacoes = 0
    for i in range(len(arr)):
        comparacoes += 1          # conta ANTES de comparar
        if arr[i] == chave:
            return i, comparacoes
    return -1, comparacoes

## 3) Fórmulas teóricas

Calculamos as curvas teóricas de cada caso para comparar com os valores empíricos.

In [ ]:
ns = np.array(TAMANHOS)

# Teórico
teorico_melhor = np.ones_like(ns, dtype=float)           # C_best = 1  (constante)
teorico_pior   = ns.astype(float)                        # C_worst = n  (linear)
teorico_medio  = P * (ns + 1) / 2 + ns * (1 - P)        # C_avg(n, p)

print(f"Para p = {P}:")
print(f"  Melhor caso: C_best = 1  (Θ(1))")
print(f"  Pior caso:   C_worst = n  (Θ(n))")
formula_medio = "p(n+1)/2 + n(1-p)"
print(f"  Caso médio:  C_avg = {formula_medio}")
print(f"  → Para p=1: C_avg = (n+1)/2  ≈  n/2")

## 4) Contagem empírica de comparações

Executamos a busca nos três cenários reais e registramos o número de comparações para validar as fórmulas:

In [ ]:
REPETICOES_MEDIO = 200   # amostras para estimar o caso médio

emp_melhor = []
emp_pior   = []
emp_medio  = []

for n in TAMANHOS:
    arr = list(range(n))   # elementos 0..n-1 distintos

    # --- Melhor caso: chave na posição 0
    _, c = busca_sequencial(arr, arr[0])
    emp_melhor.append(c)

    # --- Pior caso: chave ausente (n está fora do array)
    _, c = busca_sequencial(arr, n)
    emp_pior.append(c)

    # --- Caso médio (p=1): item sempre presente, posição uniforme
    total = 0
    for _ in range(REPETICOES_MEDIO):
        chave = random.randrange(n)
        _, c = busca_sequencial(arr, chave)
        total += c
    emp_medio.append(total / REPETICOES_MEDIO)

emp_melhor = np.array(emp_melhor)
emp_pior   = np.array(emp_pior)
emp_medio  = np.array(emp_medio)

print("Amostra (n=100):")
print(f"  Melhor caso empírico:  {emp_melhor[99]:.1f}  | teórico: {teorico_melhor[99]:.1f}")
print(f"  Pior caso empírico:    {emp_pior[99]:.1f}  | teórico: {teorico_pior[99]:.1f}")
print(f"  Caso médio empírico:   {emp_medio[99]:.1f}  | teórico: {teorico_medio[99]:.1f}")

## 5) Tabela resumida

In [ ]:
pontos = [1, 10, 50, 100, 200, 500]
idx = [n - 1 for n in pontos]

df = pd.DataFrame({
    "n": pontos,
    "melhor_teórico": teorico_melhor[idx].astype(int),
    "melhor_empírico": emp_melhor[idx].astype(int),
    "médio_teórico": teorico_medio[idx].round(1),
    "médio_empírico": emp_medio[idx].round(1),
    "pior_teórico": teorico_pior[idx].astype(int),
    "pior_empírico": emp_pior[idx].astype(int),
})
df

## 6) Gráfico principal: comparações × tamanho da entrada

Curvas teóricas (linhas sólidas) sobrepostas com os valores empíricos (marcadores pontilhados) para validação visual.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

# Curvas teóricas (linhas cheias)
ax.plot(ns, teorico_pior,   color="#d62728", linewidth=2,   label=r"Pior caso teórico: $C_{worst}(n)=n$")
ax.plot(ns, teorico_medio,  color="#ff7f0e", linewidth=2,   label=r"Caso médio teórico: $C_{avg}(n)=\frac{p(n+1)}{2}+n(1-p)$")
ax.plot(ns, teorico_melhor, color="#2ca02c", linewidth=2,   label=r"Melhor caso teórico: $C_{best}(n)=1$")

# Pontos empíricos (marcadores esparsos para não poluir)
passo = 20
ax.scatter(ns[::passo], emp_pior[::passo],   color="#d62728", marker="v", s=40, zorder=5, label="Pior caso empírico")
ax.scatter(ns[::passo], emp_medio[::passo],  color="#ff7f0e", marker="o", s=40, zorder=5, label="Caso médio empírico")
ax.scatter(ns[::passo], emp_melhor[::passo], color="#2ca02c", marker="^", s=40, zorder=5, label="Melhor caso empírico")

# Anotações nas extremidades da curva do pior caso
ax.annotate(f"n = {ns[-1]}, C = {teorico_pior[-1]:.0f}",
            xy=(ns[-1], teorico_pior[-1]),
            xytext=(ns[-1] - 80, teorico_pior[-1] - 60),
            arrowprops=dict(arrowstyle="->", color="#d62728"),
            color="#d62728", fontsize=9)

ax.annotate(f"C_avg ≈ n/2",
            xy=(ns[-1], teorico_medio[-1]),
            xytext=(ns[-1] - 120, teorico_medio[-1] + 40),
            arrowprops=dict(arrowstyle="->", color="#ff7f0e"),
            color="#ff7f0e", fontsize=9)

ax.set_xlabel("Tamanho da entrada (n)", fontsize=12)
ax.set_ylabel("Número de comparações", fontsize=12)
ax.set_title("Busca Sequencial — Melhor Caso, Caso Médio e Pior Caso\n"
             f"(p = {P}, médio estimado com {REPETICOES_MEDIO} amostras por n)", fontsize=13)
ax.legend(loc="upper left", fontsize=9)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## 7) Explorando diferentes valores de p

Altere o valor de `p` abaixo para ver como o **caso médio** se desloca entre o melhor e o pior caso.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

valores_p = [0.0, 0.25, 0.50, 0.75, 1.0]
cores_p   = ["#9467bd", "#8c564b", "#e377c2", "#bcbd22", "#17becf"]

# Pior e melhor como faixa de referência
ax.fill_between(ns, teorico_melhor, teorico_pior,
                alpha=0.07, color="gray", label="Faixa possível (melhor → pior)")
ax.plot(ns, teorico_pior,   color="#d62728", linewidth=1.8, linestyle="--", label="Pior caso (p=qualquer): C = n")
ax.plot(ns, teorico_melhor, color="#2ca02c", linewidth=1.8, linestyle="--", label="Melhor caso (p=qualquer): C = 1")

for p_val, cor in zip(valores_p, cores_p):
    c_avg = p_val * (ns + 1) / 2 + ns * (1 - p_val)
    ax.plot(ns, c_avg, color=cor, linewidth=1.8,
            label=f"Caso médio  p = {p_val:.2f}:  C_avg = {p_val:.2f}·(n+1)/2 + n·{1-p_val:.2f}")

ax.set_xlabel("Tamanho da entrada (n)", fontsize=12)
ax.set_ylabel("Número de comparações", fontsize=12)
ax.set_title("Caso médio da Busca Sequencial para diferentes valores de p", fontsize=13)
ax.legend(fontsize=8, loc="upper left")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## Conclusão

| Caso | Custo | Classe | Interpretação |
|------|-------|--------|---------------|
| **Melhor** | $C_{best}(n) = 1$ | $\Theta(1)$ | Item encontrado na 1ª posição |
| **Médio** (p=1) | $C_{avg}(n) = \dfrac{n+1}{2}$ | $\Theta(n)$ | Posição uniforme, busca sempre bem-sucedida |
| **Pior** | $C_{worst}(n) = n$ | $\Theta(n)$ | Item ausente ou na última posição |

> **Observação importante**: o caso médio e o pior caso pertencem à mesma classe $\Theta(n)$, porém o caso médio executa cerca de **metade** das comparações do pior caso.
> O melhor caso é $\Theta(1)$ — constante — mas não pode ser garantido na prática.